# LeetCode 455 — Assign Cookies bằng Python

**Link đề:** https://leetcode.com/problems/assign-cookies/
**Độ khó:** Easy · **Chủ đề:** Greedy (tham lam), Sorting, Two Pointers

```python
class Solution:
    def findContentChildren(self, g: list[int], s: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài toán ghép cặp "trẻ — bánh" sao cho số trẻ hài lòng là lớn nhất.
2. Ôn `sorted`/`list.sort`, `len`, vòng `while` với **hai con trỏ** (two pointers).
3. Đi từ cách thử mọi phân phối (vét cạn) → mỗi trẻ tìm bánh nhỏ nhất còn dùng được `O(n·m)` → **sắp xếp + hai con trỏ** `O(n log n + m log m)`.
4. Hiểu vì sao **tham lam** đúng ở đây (lập luận "trao đổi" — exchange argument), trace từng bước, phân tích Big-O.
5. Nhận dạng pattern "ghép cặp nhỏ nhất đủ dùng", làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — chia bánh cho trẻ

- Có các trẻ, trẻ `i` có **độ tham** `g[i]`: chỉ hài lòng nếu nhận bánh kích thước **≥ `g[i]`**.
- Có các bánh, bánh `j` có kích thước `s[j]`.
- Mỗi trẻ nhận **nhiều nhất 1** bánh, mỗi bánh chỉ cho **1** trẻ.
- Trả về **số trẻ hài lòng lớn nhất** có thể.

| `g` (độ tham) | `s` (bánh) | Output | Giải thích |
|---|---|:---:|---|
| `[1,2,3]` | `[1,1]` | `1` | hai bánh cỡ 1 chỉ làm vui trẻ tham 1 |
| `[1,2]` | `[1,2,3]` | `2` | đủ bánh cho cả hai trẻ |

**Constraints:** `1 <= g.length <= 3·10^4`, `0 <= s.length <= 3·10^4` (có thể **không có bánh nào**), `1 <= g[i], s[j] <= 2^31 - 1`.

Ghi chú: bài này giống hệt LeetCode 2410 (Maximum Matching of Players With Trainers).

In [1]:
g = [1, 2, 3]
s = [1, 1]
print("Số trẻ:", len(g), "| Số bánh:", len(s))
# Trẻ tham 2 có hài lòng với bánh cỡ 1 không?
print("bánh 1 đủ cho trẻ tham 2?", 1 >= 2)
print("bánh 1 đủ cho trẻ tham 1?", 1 >= 1)

Số trẻ: 3 | Số bánh: 2
bánh 1 đủ cho trẻ tham 2? False
bánh 1 đủ cho trẻ tham 1? True


Câu hỏi đặt ra trước khi code:
- Đưa bánh to cho trẻ ít tham có "phí" không?
- Thứ tự xét trẻ / bánh có quan trọng không?
- Có cần thử **mọi** cách phân phối, hay có quy tắc đơn giản luôn tối ưu?

## 2. Nền tảng Python cần biết

### 2.1. `sorted()` và `list.sort()`

- `sorted(x)` trả về **list mới** đã sắp xếp tăng dần, không đổi `x`.
- `x.sort()` sắp xếp **ngay trong** `x` và trả về `None`.
- Thêm `reverse=True` để sắp giảm dần.

Sắp xếp trong Python (Timsort) tốn `O(k log k)` với `k` phần tử.

In [2]:
a = [3, 1, 2]
b2 = sorted(a)
print("a gốc:", a, "| sorted(a):", b2)
a.sort()
print("sau a.sort():", a)
print("a.sort() trả về:", [3, 1].sort())   # None -> đừng viết a = a.sort()
print("giảm dần:", sorted([3, 1, 2], reverse=True))

a gốc: [3, 1, 2] | sorted(a): [1, 2, 3]
sau a.sort(): [1, 2, 3]
a.sort() trả về: None
giảm dần: [3, 2, 1]


### 2.2. Hai con trỏ (two pointers) trên hai list đã sắp xếp

Ta giữ hai chỉ số: `i` trỏ vào trẻ hiện tại trong `g`, `j` trỏ vào bánh hiện tại trong `s`. Mỗi bước, **ít nhất một** con trỏ tiến lên, nên vòng lặp chạy không quá `len(g) + len(s)` lần.

Điều kiện dừng `while i < len(g) and j < len(s)`: hết trẻ **hoặc** hết bánh là dừng. `and` trong Python **đánh giá ngắn mạch** (short-circuit): nếu vế trái `False` thì không xét vế phải.

In [3]:
A = [1, 4, 6]
Bv = [2, 3, 7]
i = j = 0
while i < len(A) and j < len(Bv):
    print(f"i={i} (A={A[i]}), j={j} (B={Bv[j]})", end=" -> ")
    if A[i] < Bv[j]:
        print("tăng i"); i += 1
    else:
        print("tăng j"); j += 1
print("Dừng ở i =", i, ", j =", j)

i=0 (A=1), j=0 (B=2) -> tăng i
i=1 (A=4), j=0 (B=2) -> tăng j
i=1 (A=4), j=1 (B=3) -> tăng j
i=1 (A=4), j=2 (B=7) -> tăng i
i=2 (A=6), j=2 (B=7) -> tăng i
Dừng ở i = 3 , j = 2


### 2.3. Tham lam (greedy) là gì?

**Tham lam** = ở mỗi bước chọn phương án "tốt nhất trước mắt" theo một quy tắc đơn giản, không quay lui. Tham lam **không phải lúc nào cũng đúng** — ta phải **chứng minh** quy tắc đó không làm mất nghiệm tối ưu. Bài này là ví dụ nhập môn rất đẹp cho việc chứng minh tham lam.

## 3. Cách 1 — Vét cạn (để hiểu bài, không dùng để nộp)

Thử mọi cách gán: mỗi bánh hoặc bỏ đi, hoặc cho một trẻ chưa có bánh. Ta đệ quy và lấy số trẻ hài lòng lớn nhất. Số khả năng bùng nổ theo cấp số mũ → chỉ chạy được với input rất nhỏ, nhưng nó là **oracle** (đáp án chắc chắn đúng) để kiểm tra các cách nhanh hơn.

In [4]:
def content_bruteforce(g: list[int], s: list[int]) -> int:
    n = len(g)
    best = 0

    def go(j, used, count):
        nonlocal best
        if j == len(s):
            best = max(best, count)
            return
        go(j + 1, used, count)                 # lựa chọn 1: bỏ bánh j
        for i in range(n):                     # lựa chọn 2: cho bánh j cho trẻ i
            if not used[i] and s[j] >= g[i]:
                used[i] = True
                go(j + 1, used, count + 1)
                used[i] = False

    go(0, [False] * n, 0)
    return best

print(content_bruteforce([1, 2, 3], [1, 1]))   # 1
print(content_bruteforce([1, 2], [1, 2, 3]))   # 2

1
2


Giải thích cú pháp mới:
- **Hàm lồng nhau** `go` định nghĩa bên trong hàm khác, đọc được biến của hàm ngoài.
- `nonlocal best`: cho phép hàm trong **gán lại** biến `best` của hàm ngoài (nếu không có, Python sẽ tạo biến cục bộ mới).
- Đệ quy: `go` tự gọi chính nó với `j + 1`; điểm dừng là `j == len(s)`.
- `used[i] = True` rồi `False` lại sau khi gọi: kỹ thuật **quay lui** (backtracking).

In [5]:
def demo_nonlocal():
    total = 0
    def add(x):
        nonlocal total
        total += x
    add(3); add(4)
    return total

print(demo_nonlocal())   # 7

7


## 4. Cách 2 — Mỗi trẻ (ít tham trước) chọn bánh **nhỏ nhất còn đủ**

Ý tưởng tự nhiên hơn: xét trẻ từ ít tham đến tham nhiều; mỗi trẻ lấy chiếc bánh nhỏ nhất (chưa dùng) mà vẫn làm trẻ hài lòng. Tìm bánh đó bằng cách quét toàn bộ `s` → `O(n·m)`.

In [6]:
def content_smallest_fit(g: list[int], s: list[int]) -> int:
    used = [False] * len(s)
    count = 0
    for greed in sorted(g):
        best_j = -1
        for j in range(len(s)):
            if not used[j] and s[j] >= greed and (best_j == -1 or s[j] < s[best_j]):
                best_j = j
        if best_j != -1:
            used[best_j] = True
            count += 1
    return count

print(content_smallest_fit([1, 2, 3], [1, 1]))
print(content_smallest_fit([1, 2], [1, 2, 3]))
print(content_smallest_fit([10, 9, 8, 7], [5, 6, 7, 8]))  # 2

1
2
2


### 4.1. Nút thắt

Với `n = m = 3·10^4`, `n·m ≈ 9·10^8` phép so sánh — quá chậm. Mỗi trẻ lại quét lại từ đầu những chiếc bánh đã biết chắc là quá nhỏ.

**Quan sát:** nếu **sắp xếp cả bánh**, chiếc bánh quá nhỏ cho trẻ hiện tại thì cũng quá nhỏ cho **mọi trẻ tham hơn** phía sau → có thể **bỏ hẳn** nó, không bao giờ cần quay lại. Đó là lý do một con trỏ `j` chỉ cần đi tới, không lùi.

In [7]:
g_sorted = sorted([1, 2, 3])
s_sorted = sorted([1, 1])
print("trẻ:", g_sorted, "| bánh:", s_sorted)
# Bánh cỡ 1 quá nhỏ cho trẻ tham 2 -> cũng quá nhỏ cho trẻ tham 3
print([1 >= x for x in [2, 3]])

trẻ: [1, 2, 3] | bánh: [1, 1]
[False, False]


### 4.2. Insight — tham lam bằng hai con trỏ

Sắp xếp `g` và `s` tăng dần. Đặt `i = 0` (trẻ ít tham nhất chưa được phục vụ), `j = 0` (bánh nhỏ nhất chưa dùng):

- Nếu `s[j] >= g[i]`: bánh `j` làm trẻ `i` hài lòng → **ghép**, `i += 1`, `j += 1`.
- Ngược lại: bánh `j` quá nhỏ cho trẻ `i` (và cho mọi trẻ sau) → **bỏ bánh**, `j += 1`.

Kết thúc khi hết trẻ hoặc hết bánh; đáp án là `i` (số trẻ đã được ghép).

Hai quy tắc tham lam ẩn bên trong:
1. Phục vụ **trẻ ít tham nhất trước** (dễ làm vui nhất).
2. Dùng **bánh nhỏ nhất đủ dùng** (giữ bánh to cho trẻ tham hơn).

## 5. Lời giải chính — mổ xẻ từng đoạn

### 5.1. Sắp xếp
```python
g.sort(); s.sort()
```
Sau bước này, thứ tự "nhỏ → lớn" cho phép ta chỉ đi một chiều.

### 5.2. Vòng lặp hai con trỏ
```python
i = j = 0
while i < len(g) and j < len(s):
    if s[j] >= g[i]:
        i += 1          # trẻ i đã hài lòng
    j += 1              # bánh j đã dùng hoặc bị bỏ, đằng nào cũng qua
return i
```
Mẹo gọn: `j += 1` luôn xảy ra (dù ghép hay bỏ), chỉ `i` tăng có điều kiện.

In [8]:
def content_greedy(g: list[int], s: list[int]) -> int:
    g = sorted(g)       # dùng sorted để không sửa input của người gọi
    s = sorted(s)
    i = j = 0
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:
            i += 1
        j += 1
    return i

print(content_greedy([1, 2, 3], [1, 1]))        # 1
print(content_greedy([1, 2], [1, 2, 3]))        # 2
print(content_greedy([10, 9, 8, 7], [5, 6, 7, 8]))  # 2
print(content_greedy([1, 2, 3], []))            # 0 - không có bánh

1
2
2
0


## 6. TRACE từng bước

Ví dụ: `g = [7, 8, 9, 10]`, `s = [5, 6, 7, 8]` (đã sắp xếp sẵn).

| bước | `i` | `g[i]` | `j` | `s[j]` | `s[j] >= g[i]`? | Hành động | đã vui |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 7 | 0 | 5 | Không | bỏ bánh 5 | 0 |
| 2 | 0 | 7 | 1 | 6 | Không | bỏ bánh 6 | 0 |
| 3 | 0 | 7 | 2 | 7 | Có | trẻ 7 nhận bánh 7 | 1 |
| 4 | 1 | 8 | 3 | 8 | Có | trẻ 8 nhận bánh 8 | 2 |
| — | 2 | 9 | 4 | — | hết bánh | dừng | **2** |

Ý nghĩa cột: `i` cũng chính là "số trẻ đã vui" vì trẻ được phục vụ theo thứ tự. Bánh bị bỏ là bánh quá nhỏ so với trẻ ít tham nhất còn lại.

In [9]:
def trace_content(g, s):
    g, s = sorted(g), sorted(s)
    print("g đã sắp:", g, "| s đã sắp:", s)
    i = j = 0
    step = 1
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:
            print(f"bước {step}: trẻ g[{i}]={g[i]} nhận bánh s[{j}]={s[j]}  -> vui = {i + 1}")
            i += 1
        else:
            print(f"bước {step}: bánh s[{j}]={s[j]} < g[{i}]={g[i]} -> bỏ bánh")
        j += 1
        step += 1
    reason = "hết trẻ" if i == len(g) else "hết bánh"
    print(f"Dừng vì {reason}. Kết quả = {i}")
    return i

trace_content([10, 9, 8, 7], [5, 6, 7, 8])

g đã sắp: [7, 8, 9, 10] | s đã sắp: [5, 6, 7, 8]
bước 1: bánh s[0]=5 < g[0]=7 -> bỏ bánh
bước 2: bánh s[1]=6 < g[0]=7 -> bỏ bánh
bước 3: trẻ g[0]=7 nhận bánh s[2]=7  -> vui = 1
bước 4: trẻ g[1]=8 nhận bánh s[3]=8  -> vui = 2
Dừng vì hết bánh. Kết quả = 2


2

### 6.1. Trace ca "dừng vì hết trẻ"

`g = [1, 2]`, `s = [1, 2, 3]`: hai trẻ được phục vụ, bánh cỡ 3 còn thừa.

In [10]:
trace_content([1, 2], [1, 2, 3])
print()
trace_content([1, 2, 3], [1, 1])

g đã sắp: [1, 2] | s đã sắp: [1, 2, 3]
bước 1: trẻ g[0]=1 nhận bánh s[0]=1  -> vui = 1
bước 2: trẻ g[1]=2 nhận bánh s[1]=2  -> vui = 2
Dừng vì hết trẻ. Kết quả = 2

g đã sắp: [1, 2, 3] | s đã sắp: [1, 1]
bước 1: trẻ g[0]=1 nhận bánh s[0]=1  -> vui = 1
bước 2: bánh s[1]=1 < g[1]=2 -> bỏ bánh
Dừng vì hết bánh. Kết quả = 1


1

## 7. Cách 3 — Duyệt ngược: bánh to nhất cho trẻ tham nhất còn ăn được

Một góc nhìn đối xứng: đi từ **tham nhất** xuống. Lấy bánh **to nhất** còn lại; xét trẻ tham nhất chưa xét:
- nếu bánh to nhất đủ cho trẻ đó → ghép, chuyển sang bánh to kế tiếp;
- nếu không đủ → trẻ này **không ai làm vui được** (bánh to nhất còn không đủ), bỏ trẻ.

Cũng `O(n log n + m log m)` và cho cùng đáp án. Nó minh họa: tham lam có thể đi theo nhiều hướng miễn là chứng minh được.

In [11]:
def content_greedy_reverse(g: list[int], s: list[int]) -> int:
    g = sorted(g)
    s = sorted(s)
    i = len(g) - 1        # trẻ tham nhất
    j = len(s) - 1        # bánh to nhất
    count = 0
    while i >= 0 and j >= 0:
        if s[j] >= g[i]:
            count += 1
            j -= 1        # bánh đã dùng
        i -= 1            # trẻ đã xét (được vui hoặc bị bỏ)
    return count

print(content_greedy_reverse([1, 2, 3], [1, 1]))
print(content_greedy_reverse([1, 2], [1, 2, 3]))
print(content_greedy_reverse([10, 9, 8, 7], [5, 6, 7, 8]))

1
2
2


## 8. Cách 4 (gợi mở) — đếm theo bánh bằng `bisect`

Thêm một công cụ thư viện chuẩn: `bisect.bisect_left(a, x)` tìm vị trí đầu tiên trong list **đã sắp xếp** `a` mà `a[pos] >= x`, bằng tìm kiếm nhị phân `O(log k)`.

Ta có thể duyệt trẻ (đã sắp), mỗi trẻ dùng `bisect_left` tìm bánh nhỏ nhất `>= g[i]` **kể từ vị trí con trỏ hiện tại** (`lo`). Cách này nhảy qua nhiều bánh nhỏ một lúc; hữu ích khi số bánh lớn hơn rất nhiều số trẻ.

In [12]:
import bisect
a = [1, 3, 3, 7, 9]
print(bisect.bisect_left(a, 3))   # 1: vị trí đầu tiên có giá trị >= 3
print(bisect.bisect_left(a, 4))   # 3: a[3] = 7 là số đầu tiên >= 4
print(bisect.bisect_left(a, 10))  # 5 = len(a): không có số nào >= 10

def content_bisect(g, s):
    g, s = sorted(g), sorted(s)
    pos = 0
    count = 0
    for greed in g:
        pos = bisect.bisect_left(s, greed, pos)   # chỉ tìm từ pos trở đi
        if pos == len(s):
            break
        count += 1
        pos += 1
    return count

print(content_bisect([1, 2, 3], [1, 1]), content_bisect([1, 2], [1, 2, 3]))

1
3
5
1 2


## 9. Code nộp LeetCode (Python 3)

In [13]:
class Solution:
    def findContentChildren(self, g: list[int], s: list[int]) -> int:
        g.sort()
        s.sort()
        i = j = 0
        while i < len(g) and j < len(s):
            if s[j] >= g[i]:
                i += 1
            j += 1
        return i

sol = Solution()
print(sol.findContentChildren([1, 2, 3], [1, 1]))   # 1
print(sol.findContentChildren([1, 2], [1, 2, 3]))   # 2

1
2


## 10. Vì sao tham lam đúng? (exchange argument)

Gọi OPT là một cách ghép tối ưu bất kỳ. Ta chứng minh **luôn có một cách tối ưu chứa bước tham lam đầu tiên**, rồi lặp lại cho phần còn lại.

**Bổ đề 1 — trẻ ít tham nhất `g[0]`:** nếu có bất kỳ trẻ nào được vui thì tồn tại nghiệm tối ưu trong đó trẻ `g[0]` được vui. Nếu OPT làm vui trẻ `t` nhưng không làm vui `g[0]`, ta chuyển bánh của `t` sang `g[0]` (vì `g[0] <= g[t]` nên vẫn đủ) → số trẻ vui không đổi.

**Bổ đề 2 — bánh nhỏ nhất đủ dùng `c`:** giả sử OPT đưa cho `g[0]` bánh `d` khác `c` (với `c <= d`). Hoán đổi: `g[0]` nhận `c` (vẫn đủ vì `c >= g[0]`); nếu `c` đang ở trẻ `u` thì `u` nhận `d` (đủ vì `d >= c >= g[u]`). Số trẻ vui không đổi.

**Bổ đề 3 — bánh bị bỏ:** nếu `s[j] < g[i]` với `g[i]` là trẻ ít tham nhất còn lại thì `s[j]` không làm vui được ai còn lại → bỏ nó không mất gì.

**Invariant của vòng lặp:** tại mọi thời điểm, `i` trẻ đầu tiên đã được ghép, và tồn tại nghiệm tối ưu khớp với các quyết định đã làm. Khi dừng, không thể ghép thêm → `i` là tối ưu.

## 11. Phân tích Time / Space

Gọi `n = len(g)`, `m = len(s)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Vét cạn đệ quy | mũ (≥ `2^m`) | `O(n + m)` | chỉ dùng làm oracle |
| 2. Mỗi trẻ quét bánh nhỏ nhất đủ | `O(n log n + n·m)` | `O(m)` | TLE ở cỡ lớn |
| 3. Sắp xếp + hai con trỏ | `O(n log n + m log m)` | `O(1)`* | **lời giải chuẩn** |
| 4. Duyệt ngược | `O(n log n + m log m)` | `O(1)`* | tương đương |
| 5. `bisect` | `O(n log n + m log m)` | `O(1)`* | tìm nhị phân, nhảy xa |

\* Không tính bộ nhớ của thuật toán sắp xếp (Timsort cần tới `O(n)` phụ trong trường hợp xấu). Phần hai con trỏ chỉ tốn `O(n + m)` thời gian; sắp xếp chiếm phần lớn.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", [1, 2, 3], [1, 1], 1),
    ("đề 2", [1, 2], [1, 2, 3], 2),
    ("không có bánh", [1, 2, 3], [], 0),
    ("bánh đều quá nhỏ", [5, 6], [1, 2, 3, 4], 0),
    ("chưa sắp xếp", [10, 9, 8, 7], [5, 6, 7, 8], 2),
    ("độ tham trùng", [2, 2, 2], [2, 2], 2),
    ("bánh rất to", [1, 2, 3], [2**31 - 1], 1),
    ("một trẻ một bánh", [1], [1], 1),
]
for name, g, s, expected in cases:
    check(name, Solution().findContentChildren(list(g), list(s)), expected)

PASS  đề 1: 1
PASS  đề 2: 2
PASS  không có bánh: 0
PASS  bánh đều quá nhỏ: 0
PASS  chưa sắp xếp: 2
PASS  độ tham trùng: 2
PASS  bánh rất to: 1
PASS  một trẻ một bánh: 1


In [15]:
import random
random.seed(455)
for _ in range(400):
    g = [random.randint(1, 6) for _ in range(random.randint(1, 5))]
    s = [random.randint(1, 6) for _ in range(random.randint(0, 5))]
    expected = content_bruteforce(g, s)
    for f in (content_smallest_fit, content_greedy, content_greedy_reverse, content_bisect):
        assert f(g, s) == expected, (f.__name__, g, s)
print("400 bộ ngẫu nhiên: 4 cách nhanh khớp với vét cạn")

400 bộ ngẫu nhiên: 4 cách nhanh khớp với vét cạn


### 12.1. Phản ví dụ: tham lam **sai hướng** thì sai

Quy tắc "trẻ ít tham nhất lấy bánh **to nhất**" nghe hào phóng nhưng phí bánh. Ví dụ `g = [1, 3]`, `s = [1, 3]`: đưa bánh 3 cho trẻ tham 1 → trẻ tham 3 chỉ còn bánh 1 → chỉ 1 trẻ vui, trong khi tối ưu là 2.

In [16]:
def wrong_give_largest(g, s):
    g = sorted(g)
    s = sorted(s, reverse=True)      # SAI: phát bánh to nhất trước cho trẻ ít tham
    count = 0
    remaining = list(s)
    for greed in g:
        if remaining and remaining[0] >= greed:
            remaining.pop(0)
            count += 1
    return count

print("tham lam sai hướng:", wrong_give_largest([1, 3], [1, 3]))
print("đúng              :", content_greedy([1, 3], [1, 3]))

tham lam sai hướng: 1
đúng              : 2


## 13. Lỗi người mới hay mắc

### Lỗi 1: quên sắp xếp
Hai con trỏ chỉ đúng khi cả hai list đã sắp xếp. Bỏ sắp xếp thì một bánh to nằm sớm có thể bị "bỏ" oan hoặc trẻ tham đứng đầu làm kẹt.

In [17]:
def wrong_no_sort(g, s):
    i = j = 0
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:
            i += 1
        j += 1
    return i

print("không sắp xếp:", wrong_no_sort([10, 9, 8, 7], [5, 6, 7, 8]))  # 0
print("đúng         :", content_greedy([10, 9, 8, 7], [5, 6, 7, 8]))  # 2

không sắp xếp: 0
đúng         : 2


### Lỗi 2: tăng `i` khi bánh quá nhỏ (bỏ trẻ thay vì bỏ bánh)

Nếu `s[j] < g[i]` mà ta lại `i += 1`, ta bỏ qua một trẻ dễ tính trong khi bánh nhỏ ấy vốn vô dụng → kết quả có thể thấp hơn tối ưu.

In [18]:
def wrong_skip_child(g, s):
    g, s = sorted(g), sorted(s)
    i = j = 0
    count = 0
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:
            count += 1
            j += 1
        i += 1                     # SAI: luôn bỏ trẻ, kể cả khi bánh quá nhỏ
    return count

print("sai :", wrong_skip_child([2, 3], [1, 2, 3]))   # bánh 1 làm trẻ 2 bị bỏ
print("đúng:", content_greedy([2, 3], [1, 2, 3]))

sai : 0
đúng: 2


### Lỗi 3: viết `g = g.sort()`
`list.sort()` trả `None` → `g` thành `None` và `len(g)` báo `TypeError`. Hoặc dùng `g.sort()` riêng một dòng, hoặc `g = sorted(g)`.

### Lỗi 4: quên trường hợp `s` rỗng
Với cách hai con trỏ, `while` không chạy và trả `0` — đúng. Nhưng nếu viết kiểu `s[0]` trước vòng lặp sẽ `IndexError`.

### Lỗi 5: dùng `>` thay vì `>=`
Đề nói `s[j] >= g[i]` là đủ. Dùng `>` sẽ làm sai ca `g = [1]`, `s = [1]` (đáp án 1).

In [19]:
try:
    g = [3, 1]
    g = g.sort()
    print(len(g))
except TypeError as e:
    print("TypeError:", e)

TypeError: object of type 'NoneType' has no len()


## 14. Mẹo nhận dạng pattern

Dấu hiệu "**ghép mỗi phần tử bên A với tối đa một phần tử bên B, điều kiện so sánh kiểu `b >= a`, tối đa số cặp**" → nghĩ ngay:
1. Sắp xếp cả hai phía.
2. Hai con trỏ, ghép "nhỏ nhất đủ dùng".
3. Chứng minh bằng exchange argument.

Bài cùng họ: 2410 (Players with Trainers — giống hệt), 881 Boats to Save People, 870 Advantage Shuffle, 1029 Two City Scheduling, 135 Candy (tham lam hai chiều), 2037 Minimum Number of Moves to Seat Everyone.

## 15. Biến thể

### 15.1. Trả về **danh sách cặp** (độ tham, cỡ bánh) đã ghép

In [20]:
def content_pairs(g, s):
    g, s = sorted(g), sorted(s)
    i = j = 0
    pairs = []
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:
            pairs.append((g[i], s[j]))
            i += 1
        j += 1
    return pairs

print(content_pairs([1, 2, 3], [1, 3, 2, 5]))

[(1, 1), (2, 2), (3, 3)]


### 15.2. Mỗi trẻ cần **đúng** cỡ bánh (bằng nhau) thay vì "≥"

Khi điều kiện là `s[j] == g[i]`, đáp án chỉ là đếm giao giữa hai đa tập hợp — dùng `collections.Counter` và phép `&` (giao, lấy min số lần).

In [21]:
from collections import Counter
cg = Counter([1, 2, 2, 3])
cs = Counter([2, 2, 2, 4])
print("giao:", cg & cs, "-> số cặp =", sum((cg & cs).values()))

giao: Counter({2: 2}) -> số cặp = 2


## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Đáp án ở mục 17.

### Bài 1 — Tính tay
`g = [4, 1, 3]`, `s = [3, 1, 2, 5]`. Sắp xếp và lập bảng trace hai con trỏ. Đáp án bao nhiêu?

### Bài 2 — Điền chỗ trống

In [22]:
def ex2_content(g: list[int], s: list[int]) -> int:
    # TODO 1: sắp xếp g và s
    # TODO 2: i = j = 0
    # TODO 3: while còn trẻ và còn bánh: nếu bánh đủ thì tăng i; luôn tăng j
    # TODO 4: trả i
    pass

# assert ex2_content([1, 2, 3], [1, 1]) == 1
# assert ex2_content([1, 2], [1, 2, 3]) == 2

### Bài 3 — Viết lại theo hướng duyệt ngược (từ tham nhất xuống)

In [23]:
def ex3_content_reverse(g: list[int], s: list[int]) -> int:
    # TODO: con trỏ từ cuối mỗi list đã sắp xếp
    pass

# assert ex3_content_reverse([10, 9, 8, 7], [5, 6, 7, 8]) == 2

### Bài 4 — Biến thể Boats to Save People (LeetCode 881)
Mỗi thuyền chở tối đa **2 người** với tổng cân nặng `<= limit`. Trả về **số thuyền ít nhất** để chở hết `people`. Ví dụ `people = [3, 2, 2, 1], limit = 3` → `3`.

In [24]:
def ex4_num_boats(people: list[int], limit: int) -> int:
    # TODO: sắp xếp, ghép người nặng nhất với người nhẹ nhất nếu được
    pass

# assert ex4_num_boats([1, 2], 3) == 1
# assert ex4_num_boats([3, 2, 2, 1], 3) == 3
# assert ex4_num_boats([3, 5, 3, 4], 5) == 4

### Bài 5 — Mini challenge: mỗi trẻ được tối đa **2 bánh**, hài lòng nếu tổng ≥ độ tham
Viết hàm vét cạn nhỏ (input ≤ 6 phần tử) để tìm số trẻ vui tối đa, rồi thử xem quy tắc hai con trỏ cũ **còn đúng không** bằng một phản ví dụ. (Bài này dạy: đổi đề một chút là tham lam cũ có thể sai.)

In [25]:
def ex5_content_two_cookies_bruteforce(g: list[int], s: list[int]) -> int:
    # TODO: thử mọi cách gán mỗi bánh cho một trẻ hoặc bỏ đi, mỗi trẻ <= 2 bánh
    pass

# assert ex5_content_two_cookies_bruteforce([3], [1, 2]) == 1

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: sắp `g = [1,3,4]`, `s = [1,2,3,5]`. Mức 2: bánh 1 cho trẻ 1; bánh 2 < 3 bỏ. Mức 3: bánh 3 cho trẻ 3, bánh 5 cho trẻ 4.

**Bài 2** — Mức 1: hai list phải sắp xếp. Mức 2: điều kiện `while i < len(g) and j < len(s)`. Mức 3: chỉ `i += 1` trong `if s[j] >= g[i]`, `j += 1` ngoài `if`.

**Bài 3** — Mức 1: bắt đầu `i = len(g) - 1`, `j = len(s) - 1`. Mức 2: bánh to nhất không đủ thì bỏ **trẻ**. Mức 3: ghép thì giảm cả `i` và `j`, đếm `count`.

**Bài 4** — Mức 1: người nặng nhất luôn phải đi một thuyền. Mức 2: ghép được thì ghép với người nhẹ nhất (`lo`). Mức 3: `while lo <= hi:` mỗi vòng `hi -= 1`, nếu `people[lo] + people[hi] <= limit` thì `lo += 1`; đếm số thuyền.

**Bài 5** — Mức 1: vét cạn: mỗi bánh có `n + 1` lựa chọn (cho trẻ nào hoặc bỏ). Mức 2: lưu `sum_[i]`, `cnt[i]` cho mỗi trẻ. Mức 3: thử `g = [2, 2]`, `s = [1, 1, 2]`.

## 17. Đáp án tham khảo

**Bài 1:** `g = [1,3,4]`, `s = [1,2,3,5]`: (1,1) ghép; 2 < 3 bỏ; (3,3) ghép; (4,5) ghép → **3**.

In [26]:
def ans2_content(g, s):
    g, s = sorted(g), sorted(s)
    i = j = 0
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:
            i += 1
        j += 1
    return i

def ans3_content_reverse(g, s):
    g, s = sorted(g), sorted(s)
    i, j, count = len(g) - 1, len(s) - 1, 0
    while i >= 0 and j >= 0:
        if s[j] >= g[i]:
            count += 1
            j -= 1
        i -= 1
    return count

def ans4_num_boats(people, limit):
    people = sorted(people)
    lo, hi = 0, len(people) - 1
    boats = 0
    while lo <= hi:
        if people[lo] + people[hi] <= limit:
            lo += 1
        hi -= 1
        boats += 1
    return boats

def ans5_content_two_cookies_bruteforce(g, s):
    n = len(g)
    total = [0] * n
    cnt = [0] * n
    best = 0
    def go(j):
        nonlocal best
        if j == len(s):
            best = max(best, sum(1 for i in range(n) if total[i] >= g[i]))
            return
        go(j + 1)
        for i in range(n):
            if cnt[i] < 2:
                total[i] += s[j]; cnt[i] += 1
                go(j + 1)
                total[i] -= s[j]; cnt[i] -= 1
    go(0)
    return best

assert ans2_content([4, 1, 3], [3, 1, 2, 5]) == 3            # bài 1
assert ans2_content([1, 2, 3], [1, 1]) == 1
assert ans2_content([1, 2], [1, 2, 3]) == 2
assert ans3_content_reverse([10, 9, 8, 7], [5, 6, 7, 8]) == 2
assert ans4_num_boats([1, 2], 3) == 1
assert ans4_num_boats([3, 2, 2, 1], 3) == 3
assert ans4_num_boats([3, 5, 3, 4], 5) == 4
assert ans5_content_two_cookies_bruteforce([3], [1, 2]) == 1
# Phản ví dụ bài 5: hai con trỏ kiểu cũ cho 1, vét cạn cho 2
print("bài 5 – hai con trỏ cũ:", ans2_content([2, 2], [1, 1, 2]),
      "| vét cạn 2 bánh/trẻ:", ans5_content_two_cookies_bruteforce([2, 2], [1, 1, 2]))
print("Đáp án bài 1–5: PASS")

bài 5 – hai con trỏ cũ: 1 | vét cạn 2 bánh/trẻ: 2
Đáp án bài 1–5: PASS


## 18. Template nhớ nhanh + checklist submit

```python
# Greedy ghép cặp "nhỏ nhất đủ dùng"
A.sort(); B.sort()
i = j = 0
while i < len(A) and j < len(B):
    if B[j] >= A[i]:   # ghép được
        i += 1
    j += 1             # phần tử B đã dùng hoặc vô dụng
return i
```

**Checklist trước khi submit**
- [ ] Đã sắp xếp **cả hai** list chưa?
- [ ] Điều kiện là `>=` (không phải `>`)?
- [ ] Bánh quá nhỏ thì bỏ **bánh** (tăng `j`), không bỏ trẻ?
- [ ] Ca `s = []` trả `0`?
- [ ] Không viết `g = g.sort()`?
- [ ] Giải thích được vì sao tham lam đúng (đổi chỗ không làm giảm số trẻ vui)?